# Download Kaleida Health prices and add them to the database
Kaleida's pointer file (`kaleidahealth.org/cms-hpt.txt`) lists one machine-readable file for Buffalo General, DeGraff, Golisano Children's and Millard Fillmore Suburban.

Run the cells top to bottom.

In [ ]:
import subprocess, sys, json
from pathlib import Path

DATA = Path('../data'); DATA.mkdir(exist_ok=True)
DEST = DATA / '161533232_kaleida-health_standardcharges.json'
POINTER_URL = 'https://hospitalpricedisclosure.com/download.aspx?pi=8O1flKb9ncxmFbwWjt6PBg*-*'
DIRECT_URL = 'https://cleverleypteusstatic.blob.core.windows.net/readable/161533232_kaleida-health_standardcharges.json'

## 1. Download
Uses `curl` (follows the redirect). Tries the official link first, then the direct file link.

In [ ]:
def download(url):
    tmp = DEST.with_suffix('.part')
    r = subprocess.run(['curl', '-L', '--fail', '--silent', '--show-error', '-A', 'Mozilla/5.0',
                        '-o', str(tmp), url], capture_output=True, text=True)
    if r.returncode != 0:
        print('failed:', r.stderr.strip()); return False
    tmp.rename(DEST); return True

ok = download(POINTER_URL) or download(DIRECT_URL)
print('downloaded' if ok else 'DOWNLOAD FAILED', '->', DEST, f'{DEST.stat().st_size/1e6:.1f} MB' if DEST.exists() else '')

## 2. Check the file looks right

In [ ]:
with open(DEST, 'rb') as f:
    head = f.read(600).decode('utf-8-sig', 'replace')
print(head)
assert head.lstrip().startswith('{'), 'Not JSON - the download probably returned a web page. Open the link in a browser and save the file into data/.'

## 3. Add to `truecost.db`

In [ ]:
sys.path.insert(0, '../scripts')
import add_kaleida
add_kaleida.main()

## 4. Verify

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect('../truecost.db')
pd.read_sql("select hospital, count(*) rows, count(distinct code) codes from prices group by hospital", con)